In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Base libraries loaded fine!")

Base libraries loaded fine!


In [2]:
import pyarrow
print("Pyarrow loaded fine!")

Pyarrow loaded fine!


In [2]:
# No sys.path hacks needed anymore!

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import your custom package
from cooltrack.constants import INDEPENDENT_DIMS
from cooltrack.data_loader import load_and_clean_grid_pandas
from cooltrack.models import ThermalEvolutionModels
from cooltrack.integrator import CoolingIntegrator

In [ ]:

# Adjust this path based on where the notebook is relative to your data
GRID_FILE_PATH = "../../data/HADES_grid/hades_processed_grid.parquet"

In [ ]:
print("Using src.data_loader to fetch data...")
# The data loader handles all the PyArrow filtering and unit conversions automatically
df = load_and_clean_grid_pandas(GRID_FILE_PATH)

# Let's use a smaller slice of the data so training is instant for our plot test
df_test = df[df['mass_Mj'] <= 10].copy()
print(f"Ready with {len(df_test):,} rows.")

In [ ]:
print("Training models using src.models.ThermalEvolutionModels...")
ml_engine = ThermalEvolutionModels()

# This trains both the T_int and dS/dt models and stores them inside ml_engine
ml_engine.train_models(df_test, tune_hyperparameters=False)

print("Models trained and ready!")

In [ ]:
print("Integrating a track using src.integrator.CoolingIntegrator...")
integrator = CoolingIntegrator(ml_engine)

# Pick a target planet to plot (e.g., 1 Jupiter Mass, 500K Irradiation)
test_mass = 1.0
test_tirr = 500.0

# Find a matching row in our dataset to get realistic fixed parameters
planet_row = df_test[(np.isclose(df_test['mass_Mj'], test_mass, atol=0.1)) & 
                     (np.isclose(df_test['T_irr'], test_tirr, atol=10))].iloc[0]

# Define hot and cold entropy boundaries
s_start = df_test['S_physical'].max() 
s_end = df_test['S_physical'].min()   

# Calculate the full track using our new method!
ages, entropies = integrator.calculate_track(planet_row, s_start, s_end)

if ages is not None:
    # We also need to predict T_int at each step to plot the physical temperature
    fixed_params = planet_row[INDEPENDENT_DIMS].values.astype(float)
    t_ints = []
    
    for s in entropies:
        # Reconstruct the feature vector: [mass, T_irr, Met, core, f_sed, kzz, S]
        inp = np.append(fixed_params, s).reshape(1, -1)
        # Use our trained T_int model to predict the temperature at this entropy
        t_ints.append(ml_engine.tint_model.predict(inp)[0])

    # --- Plotting ---
    fig, ax1 = plt.subplots(figsize=(10, 6))

    color = 'tab:red'
    ax1.set_xlabel('Age (Years)')
    ax1.set_ylabel('Internal Temperature (K)', color=color)
    ax1.plot(ages, t_ints, color=color, lw=3)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_xscale('log') 
    
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('Physical Entropy (J/K/kg)', color=color)  
    ax2.plot(ages, entropies, color=color, lw=3, linestyle='--')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f"CoolTrack: {test_mass} $M_J$, $T_{{irr}}$={test_tirr} K")
    fig.tight_layout()  
    plt.show()
else:
    print("Integration failed or returned None.")

In [ ]:
def age_ode_ml(s, t, fixed_params):
    # 1. Predict T_int
    tint_input = np.append(fixed_params, s).reshape(1, -1)
    current_tint = tint_model.predict(tint_input)[0]
    
    # 2. Predict dS/dt
    dsdt_input = np.append(tint_input[0], current_tint).reshape(1, -1)
    log_abs_dsdt = dsdt_model.predict(dsdt_input)[0]
    
    abs_dsdt = 10**log_abs_dsdt
    if abs_dsdt < 1e-20:
        return -np.inf 
        
    return -1.0 / abs_dsdt

# Define a test planet (e.g., 1 Jupiter mass, 500K irradiation)
test_mass = 1.0
test_tirr = 500.0

# Extract a valid row from the grid to get realistic initial/final entropies
planet_data = df[(np.isclose(df['mass_Mj'], test_mass, atol=0.1)) & 
                 (np.isclose(df['T_irr'], test_tirr, atol=10))].sort_values('S_physical', ascending=False)

if not planet_data.empty:
    fixed_params = planet_data.iloc[0][INDEPENDENT_DIMS].values.astype(float)
    s_start = planet_data['S_physical'].max() # Hot start
    s_end = planet_data['S_physical'].min()   # Cold end
    
    print(f"Integrating from S={s_start:.1f} to S={s_end:.1f}...")
    
    # Solve the ODE but evaluate at specific entropy steps so we get a smooth curve
    s_eval = np.linspace(s_start, s_end, 100)
    
    solution = solve_ivp(
        fun=age_ode_ml,
        t_span=[s_start, s_end],
        y0=[0],
        t_eval=s_eval,
        method='RK45',
        args=(fixed_params,)
    )
    
    if solution.status == 0:
        ages_yr = solution.y[0] / SECONDS_PER_YR
        entropies = solution.t
        
        # We also need to predict the T_int at each of these steps to plot it!
        t_ints = []
        for s in entropies:
            inp = np.append(fixed_params, s).reshape(1, -1)
            t_ints.append(tint_model.predict(inp)[0])
            
        print("Integration successful!")
    else:
        print("Integration failed:", solution.message)
else:
    print("Could not find a matching planet in the sliced data.")

In [ ]:
if solution.status == 0:
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot Age vs T_int
    color = 'tab:red'
    ax1.set_xlabel('Age (Years)')
    ax1.set_ylabel('Internal Temperature (K)', color=color)
    ax1.plot(ages_yr, t_ints, color=color, lw=3)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_xscale('log') # Age is best viewed on a log scale
    
    # Create a secondary y-axis to show Entropy dropping simultaneously
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('Physical Entropy (J/K/kg)', color=color)  
    ax2.plot(ages_yr, entropies, color=color, lw=3, linestyle='--')
    ax2.tick_params(axis='y', labelcolor=color)

    plt.title(f"Cooling Track: {test_mass} $M_J$, $T_{{irr}}$={test_tirr} K")
    fig.tight_layout()  
    plt.show()